# 🛰️ SatQuery AI — Model 3: Multi-Temporal Change Detection VQA

**Team Spectra | Smart India Hackathon 2026**

This notebook fine-tunes a **Dual-Stream BLIP-2** architecture with **QLoRA**
for Visual Question Answering on **bi-temporal image pairs** ($T_1$ before,
$T_2$ after) of the same geographic area. The model detects, classifies, and
describes land-cover changes in free-form natural language.

### Pipeline
1. Auto-collect ~100 bi-temporal pairs (EuroSAT $T_1$ + synthesised $T_2$ changes)
2. Generate ~500 change-detection QA pairs requiring temporal reasoning
3. Reuse the DualStreamBLIP2 wrapper (shared ViT → concat → LoRA LM)
4. Fine-tune with 4-bit quantization + LoRA
5. Evaluate with BLEU, ROUGE-L metrics (overall + per change type)
6. Generate visualizations & save everything to Google Drive

**Requirements:** Google Colab with T4 GPU (free tier works)

---
## 1 · Environment Setup

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
!pip install -q "transformers>=4.36.0" "peft>=0.7.0" "bitsandbytes>=0.41.0" \
    "accelerate>=0.25.0" "datasets>=2.16.0" evaluate nltk rouge-score \
    pillow matplotlib seaborn tqdm scipy

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os, json, random, csv, gc, warnings, time, copy
from io import BytesIO
from datetime import datetime
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw, ImageFilter

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import nltk
for _pkg in ("punkt", "wordnet", "punkt_tab"):
    nltk.download(_pkg, quiet=True)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 150, "font.size": 11})
print("✅ All packages imported successfully")

In [ ]:
# ── Mount Google Drive & Configuration ───────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

CONFIG = dict(
    seed            = 42,
    drive_output    = "/content/drive/MyDrive/SatQuery_AI/Model3_ChangeDetect",
    data_dir        = "/content/rs_bitemporal_data",
    num_pairs       = 102,          # ~17 per change type × 6 types
    image_size      = 224,
    model_name      = "Salesforce/blip2-opt-2.7b",
    lora_rank       = 16,
    lora_alpha      = 32,
    lora_dropout    = 0.05,
    learning_rate   = 2e-4,
    num_epochs      = 8,
    batch_size      = 2,
    grad_accum_steps= 4,
    max_length      = 256,
    warmup_ratio    = 0.1,
    val_split       = 0.2,
    early_stop_patience = 2,
)

# Create output directories
for _sub in ("checkpoints", "checkpoints/processor", "results", "dataset", "report"):
    os.makedirs(f"{CONFIG['drive_output']}/{_sub}", exist_ok=True)
for _sub in ("t1_before", "t2_after", "pairs"):
    os.makedirs(f"{CONFIG['data_dir']}/{_sub}", exist_ok=True)

# Reproducibility
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n🖥️  Device : {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU    : {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM   : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

with open(f"{CONFIG['drive_output']}/config.json", "w") as _f:
    json.dump(CONFIG, _f, indent=2)
print(f"\n📁 Output : {CONFIG['drive_output']}")
print("✅ Setup complete")

---
## 2 · Bi-Temporal Dataset Collection

| Component | Details |
|-----------|---------|
| **$T_1$ (before)** | Real Sentinel-2 EuroSAT RGB images (10 land-use classes) |
| **$T_2$ (after)** | Synthesised from $T_1$ with realistic temporal changes |

### Change Types
| Type | Description | Source Classes |
|------|-------------|---------------|
| Urbanisation | Green → built-up structures | Forest, Pasture, AnnualCrop |
| Deforestation | Dense canopy → bare earth | Forest, HerbaceousVegetation |
| Flooding | Normal terrain → water inundation | River, Pasture, AnnualCrop |
| Seasonal crop | Growing → harvested / bare soil | AnnualCrop, PermanentCrop |
| Fire / burn scar | Vegetation → charred patches | Forest, HerbaceousVegetation |
| No change | Control — minor natural variation | Any class |

In [ ]:
# ── 2a  Download T1 Images (EuroSAT) ────────────────────────────────────────
from datasets import load_dataset

print("=" * 60)
print("📡  STEP 1 : Downloading EuroSAT (T1 — Before)")
print("=" * 60)

EUROSAT_CLASSES = [
    "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway",
    "Industrial", "Pasture", "PermanentCrop", "Residential",
    "River", "SeaLake",
]

eurosat_ds = None
for _src in ("blanchon/EuroSAT", "tanganke/EuroSAT"):
    try:
        print(f"  Trying {_src} …")
        eurosat_ds = load_dataset(_src, split="train", trust_remote_code=True)
        print(f"  ✅ Loaded: {len(eurosat_ds)} images")
        break
    except Exception as _e:
        print(f"  ❌ {_e}")

# Fallback: download ZIP from Zenodo
if eurosat_ds is None:
    print("\n  Downloading from Zenodo …")
    os.system(
        "wget -q https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip "
        "-O /content/eurosat.zip && "
        "unzip -q -o /content/eurosat.zip -d /content/eurosat_raw/"
    )
    _root = Path("/content/eurosat_raw")
    _candidates = list(_root.rglob("*.jpg")) + list(_root.rglob("*.tif"))
    print(f"  Found {len(_candidates)} image files")

# Collect more images per class for diverse change scenarios
t1_samples = []
per_class = max(CONFIG["num_pairs"] // len(EUROSAT_CLASSES) + 2, 12)

if eurosat_ds is not None:
    class_counts = defaultdict(int)
    indices = list(range(len(eurosat_ds)))
    random.shuffle(indices)

    for idx in indices:
        sample = eurosat_ds[idx]
        label  = sample["label"]
        cls    = EUROSAT_CLASSES[label] if label < len(EUROSAT_CLASSES) else f"class_{label}"
        if class_counts[cls] >= per_class:
            continue
        img = sample["image"]
        if not isinstance(img, Image.Image):
            img = Image.fromarray(np.array(img))
        img = img.convert("RGB")
        _path = f"{CONFIG['data_dir']}/t1_before/{cls}_{class_counts[cls]:02d}.png"
        img.save(_path)
        t1_samples.append(dict(image_path=_path, cls=cls))
        class_counts[cls] += 1
        if len(t1_samples) >= per_class * len(EUROSAT_CLASSES):
            break
else:
    _root = Path("/content/eurosat_raw")
    for cls_dir in sorted(p for p in _root.rglob("*") if p.is_dir()):
        cls = cls_dir.name
        imgs = sorted(cls_dir.glob("*.*"))
        for i, ip in enumerate(imgs[:per_class]):
            _path = f"{CONFIG['data_dir']}/t1_before/{cls}_{i:02d}.png"
            Image.open(ip).convert("RGB").save(_path)
            t1_samples.append(dict(image_path=_path, cls=cls))

del eurosat_ds; gc.collect()
print(f"\n  Collected {len(t1_samples)} T1 (before) images across {len(set(s['cls'] for s in t1_samples))} classes")

In [ ]:
# ── 2b  Define Change Synthesis Functions ────────────────────────────────────
from scipy.ndimage import gaussian_filter

print("\n" + "=" * 60)
print("🔄  STEP 2 : Synthesising T2 (After) Images with Temporal Changes")
print("=" * 60)

# Define which classes are eligible for each change type
CHANGE_ELIGIBLE = {
    "urbanisation":    ["Forest", "Pasture", "AnnualCrop", "HerbaceousVegetation"],
    "deforestation":   ["Forest", "HerbaceousVegetation"],
    "flooding":        ["River", "Pasture", "AnnualCrop", "SeaLake"],
    "seasonal_crop":   ["AnnualCrop", "PermanentCrop", "Pasture"],
    "fire_burn":       ["Forest", "HerbaceousVegetation", "PermanentCrop"],
    "no_change":       EUROSAT_CLASSES,  # any class
}

def synthesize_urbanisation(img_arr):
    """Add rectangular building-like structures over green areas."""
    h, w = img_arr.shape[:2]
    result = img_arr.copy()
    n_buildings = random.randint(3, 8)
    for _ in range(n_buildings):
        bh = random.randint(15, 45)
        bw = random.randint(15, 45)
        y = random.randint(0, max(0, h - bh))
        x = random.randint(0, max(0, w - bw))
        # Grey concrete colour with slight variation
        grey = random.randint(140, 200)
        result[y:y+bh, x:x+bw] = [grey, grey - 10, grey - 20]
    # Add some road-like linear features
    for _ in range(random.randint(1, 3)):
        road_y = random.randint(10, h - 10)
        thickness = random.randint(3, 6)
        result[road_y:road_y+thickness, :] = [160, 155, 150]
    return result

def synthesize_deforestation(img_arr):
    """Replace green regions with brown bare soil."""
    h, w = img_arr.shape[:2]
    result = img_arr.copy()
    # Create irregular deforested patches
    n_patches = random.randint(2, 5)
    for _ in range(n_patches):
        cy, cx = random.randint(20, h-20), random.randint(20, w-20)
        ry, rx = random.randint(20, 50), random.randint(20, 50)
        Y, X = np.ogrid[:h, :w]
        mask = ((Y - cy)**2 / ry**2 + (X - cx)**2 / rx**2) < 1.0
        # Brown soil tones
        brown_r = random.randint(140, 175)
        brown_g = random.randint(100, 130)
        brown_b = random.randint(60, 90)
        result[mask] = [brown_r, brown_g, brown_b]
        # Add some lighter patches within (exposed earth)
        inner_mask = ((Y - cy)**2 / (ry*0.5)**2 + (X - cx)**2 / (rx*0.5)**2) < 1.0
        result[inner_mask] = [brown_r + 20, brown_g + 15, brown_b + 10]
    return result

def synthesize_flooding(img_arr):
    """Add dark blue water inundation patches."""
    h, w = img_arr.shape[:2]
    result = img_arr.copy()
    # Create a flood mask — floods tend to fill lower areas
    flood_mask = np.zeros((h, w), dtype=bool)
    n_zones = random.randint(2, 5)
    for _ in range(n_zones):
        cy, cx = random.randint(10, h-10), random.randint(10, w-10)
        ry, rx = random.randint(25, 60), random.randint(25, 60)
        Y, X = np.ogrid[:h, :w]
        zone = ((Y - cy)**2 / ry**2 + (X - cx)**2 / rx**2) < 1.0
        flood_mask |= zone
    # Dark blue-brown water with turbidity variation
    water_r = random.randint(30, 60)
    water_g = random.randint(50, 80)
    water_b = random.randint(90, 130)
    result[flood_mask] = [water_r, water_g, water_b]
    # Add slight gaussian blur for realistic water appearance
    for c in range(3):
        blurred = gaussian_filter(result[:,:,c].astype(float), sigma=1.5)
        result[:,:,c] = np.where(flood_mask, blurred.astype(np.uint8), result[:,:,c])
    return result

def synthesize_seasonal_crop(img_arr):
    """Change green growing crops to brown harvested / bare soil."""
    result = img_arr.astype(np.float64)
    # Reduce green channel, boost red → brown/yellow tones (harvested field)
    result[:,:,0] = np.clip(result[:,:,0] * 1.3 + 30, 0, 255)   # more red
    result[:,:,1] = np.clip(result[:,:,1] * 0.6 - 10, 0, 255)   # less green
    result[:,:,2] = np.clip(result[:,:,2] * 0.5, 0, 255)         # less blue
    # Add some stubble texture
    noise = np.random.normal(0, 8, result.shape)
    result = np.clip(result + noise, 0, 255)
    return result.astype(np.uint8)

def synthesize_fire_burn(img_arr):
    """Add dark charred patches with orange fire-edge boundaries."""
    h, w = img_arr.shape[:2]
    result = img_arr.copy()
    n_burns = random.randint(2, 4)
    for _ in range(n_burns):
        cy, cx = random.randint(20, h-20), random.randint(20, w-20)
        ry, rx = random.randint(20, 50), random.randint(20, 50)
        Y, X = np.ogrid[:h, :w]
        # Outer ring — orange/brown fire edge
        outer = ((Y - cy)**2 / (ry*1.3)**2 + (X - cx)**2 / (rx*1.3)**2) < 1.0
        inner = ((Y - cy)**2 / ry**2 + (X - cx)**2 / rx**2) < 1.0
        edge = outer & ~inner
        result[edge] = [180, 100, 30]  # orange-brown fire edge
        # Inner area — dark charred
        result[inner] = [random.randint(25, 50), random.randint(20, 35), random.randint(15, 25)]
    return result

def synthesize_no_change(img_arr):
    """Minor natural variation only — brightness/contrast jitter."""
    result = img_arr.astype(np.float64)
    # Small brightness shift
    brightness = random.uniform(-15, 15)
    result += brightness
    # Small contrast shift
    contrast = random.uniform(0.9, 1.1)
    mean = result.mean()
    result = (result - mean) * contrast + mean
    # Tiny noise
    result += np.random.normal(0, 3, result.shape)
    return np.clip(result, 0, 255).astype(np.uint8)

CHANGE_FUNCTIONS = {
    "urbanisation":  synthesize_urbanisation,
    "deforestation": synthesize_deforestation,
    "flooding":      synthesize_flooding,
    "seasonal_crop": synthesize_seasonal_crop,
    "fire_burn":     synthesize_fire_burn,
    "no_change":     synthesize_no_change,
}

In [ ]:
# ── 2c  Create Bi-Temporal Pairs ────────────────────────────────────────────
pairs_per_type = CONFIG["num_pairs"] // len(CHANGE_FUNCTIONS)  # ~17 each
bitemporal_pairs = []

# Group T1 samples by class for targeted assignment
t1_by_class = defaultdict(list)
for s in t1_samples:
    t1_by_class[s["cls"]].append(s)

pair_id = 0
for change_type, synth_fn in CHANGE_FUNCTIONS.items():
    eligible_classes = CHANGE_ELIGIBLE[change_type]
    # Gather eligible T1 images
    eligible_t1 = []
    for cls in eligible_classes:
        eligible_t1.extend(t1_by_class.get(cls, []))
    random.shuffle(eligible_t1)

    count = 0
    for t1 in eligible_t1:
        if count >= pairs_per_type:
            break

        t1_img = np.array(Image.open(t1["image_path"]).convert("RGB"))
        t2_img = synth_fn(t1_img)

        t2_path = f"{CONFIG['data_dir']}/t2_after/{change_type}_{pair_id:04d}.png"
        Image.fromarray(t2_img).save(t2_path)

        bitemporal_pairs.append(dict(
            t1_path=t1["image_path"],
            t2_path=t2_path,
            cls=t1["cls"],
            change_type=change_type,
            pair_id=pair_id,
        ))
        pair_id += 1
        count += 1

gc.collect()
print(f"\n📊 Created {len(bitemporal_pairs)} bi-temporal pairs")
_ct_counts = defaultdict(int)
for p in bitemporal_pairs:
    _ct_counts[p["change_type"]] += 1
for ct, n in sorted(_ct_counts.items()):
    print(f"   {ct:20s} : {n} pairs")

---
## 2d · Change-Detection QA Pair Generation

In [ ]:
print("\n" + "=" * 60)
print("💬  STEP 3 : Generating Change-Detection QA Pairs")
print("=" * 60)

# ── Detailed change-type metadata ────────────────────────────────────────────
CHANGE_INFO = {
    "urbanisation": dict(
        name="urbanisation",
        desc="conversion of natural or agricultural land to built-up urban area",
        t1_desc="green vegetation, open fields, or natural land cover",
        t2_desc="grey rectangular structures resembling buildings, paved road-like features, and reduced vegetation",
        extent="moderate to significant — new structures cover portions of the previously open landscape",
        direction="natural land → urban built-up area",
        temporal_cue="progressive development with new construction replacing vegetation",
        impact="loss of agricultural or natural land, increased impervious surfaces, altered local hydrology, potential urban heat island effects",
    ),
    "deforestation": dict(
        name="deforestation",
        desc="clearing of dense forest or vegetation canopy to expose bare soil",
        t1_desc="dense green tree canopy with continuous forest cover",
        t2_desc="brown bare soil patches where trees have been removed, with irregular clearing boundaries",
        extent="substantial — large patches of canopy removed exposing underlying earth",
        direction="dense forest → cleared bare land",
        temporal_cue="abrupt transition from intact canopy to exposed soil consistent with recent clearing",
        impact="habitat loss, reduced carbon sequestration, increased soil erosion risk, potential watershed degradation",
    ),
    "flooding": dict(
        name="flooding",
        desc="inundation of previously dry terrain by water from rainfall, river overflow, or storm surge",
        t1_desc="normal dry terrain with vegetation, fields, or mixed land cover",
        t2_desc="dark blue-brown water patches covering low-lying areas, submerged vegetation, turbid water surface",
        extent="widespread — water covers a significant portion of the previously dry area",
        direction="dry land → water inundation",
        temporal_cue="sudden appearance of standing water absent in the earlier observation",
        impact="displacement of communities, crop destruction, infrastructure damage, potential waterborne disease risk",
    ),
    "seasonal_crop": dict(
        name="seasonal crop change",
        desc="transition from active growing season to post-harvest or fallow state",
        t1_desc="green agricultural fields with visible crop growth and active vegetation",
        t2_desc="brown-yellow harvested or bare fields with crop stubble residue and reduced vegetation",
        extent="scene-wide — the entire agricultural area has transitioned from growing to harvested",
        direction="green growing crops → brown harvested fields",
        temporal_cue="natural seasonal progression consistent with agricultural harvest cycle",
        impact="normal agricultural practice; indicates successful harvest completion, soil now exposed to potential erosion",
    ),
    "fire_burn": dict(
        name="fire or burn scar",
        desc="destruction of vegetation by wildfire leaving charred remnants and burn scars",
        t1_desc="healthy green vegetation cover with intact canopy or ground cover",
        t2_desc="dark charred patches with orange-brown fire edge boundaries, severely damaged vegetation",
        extent="moderate — distinct burn scar areas with clear boundaries between burnt and unburnt zones",
        direction="healthy vegetation → charred burn scar",
        temporal_cue="abrupt, non-seasonal vegetation loss with characteristic burn scar patterns",
        impact="ecosystem destruction, wildlife habitat loss, air quality degradation, increased runoff and erosion risk",
    ),
    "no_change": dict(
        name="no significant change",
        desc="temporal stability with only minor natural variation between observation times",
        t1_desc="the scene shows its characteristic land cover features",
        t2_desc="the scene remains essentially unchanged with only minor brightness or contrast differences from natural variation",
        extent="negligible — no meaningful land cover change detected",
        direction="stable land cover → same land cover",
        temporal_cue="consistent appearance across both timestamps indicating temporal stability",
        impact="no significant environmental or land-use change; area remains stable",
    ),
}

CLASS_CONTEXT = {
    "AnnualCrop": "agricultural fields with annual crops",
    "Forest": "dense forest with tree canopy cover",
    "HerbaceousVegetation": "grasslands and herbaceous vegetation",
    "Highway": "highway and road infrastructure",
    "Industrial": "industrial zones with large buildings",
    "Pasture": "pastoral grazing land and meadows",
    "PermanentCrop": "permanent crop plantations and orchards",
    "Residential": "residential housing and neighbourhoods",
    "River": "river channels and riparian zones",
    "SeaLake": "large water bodies and coastal areas",
}


def generate_change_qa(pair):
    """Return 5 change-detection QA dicts for a bi-temporal pair."""
    ct   = pair["change_type"]
    cls  = pair["cls"]
    info = CHANGE_INFO[ct]
    ctx  = CLASS_CONTEXT.get(cls, cls.lower())
    qa   = []

    # Q1 – Change detection (what changed?)
    qa.append(dict(
        question="What changes occurred in this area between the two observation times?",
        answer=(f"Comparing the two timestamps, this area has experienced {info['desc']}. "
                f"In T1, the scene shows {info['t1_desc']}, characteristic of {ctx}. "
                f"In T2, the scene now shows {info['t2_desc']}. "
                f"The transition from {info['direction']} is clearly visible.")))

    # Q2 – Change classification (what type?)
    qa.append(dict(
        question="What type of land-cover change has occurred between T1 and T2?",
        answer=(f"The observed change is classified as {info['name']}. "
                f"The T1 observation shows {info['t1_desc']}, while T2 reveals "
                f"{info['t2_desc']}. This pattern of {info['direction']} "
                f"is characteristic of {info['name']} processes.")))

    # Q3 – Change extent (how much?)
    qa.append(dict(
        question="How extensive is the change between T1 and T2?",
        answer=(f"The change extent is {info['extent']}. "
                f"In T1, the original {ctx} landscape is visible. "
                f"In T2, {info['t2_desc']}. "
                f"The {info['temporal_cue']} indicates {info['name']}.")))

    # Q4 – Temporal reasoning (which is newer?)
    if ct == "no_change":
        qa.append(dict(
            question="Is there evidence that one image was taken more recently than the other?",
            answer=(f"There is no strong temporal indicator. Both T1 and T2 show {ctx} "
                    f"with {info['t2_desc']}. The minor variations in brightness and contrast "
                    f"are consistent with different atmospheric conditions or sensor calibration "
                    f"rather than actual land-cover change.")))
    else:
        qa.append(dict(
            question="Is T2 more recent than T1? What evidence suggests this?",
            answer=(f"Yes, T2 appears to be the more recent observation. "
                    f"The progression shows {info['direction']}, which follows "
                    f"a logical temporal sequence. The {info['temporal_cue']} "
                    f"confirms that T2 was acquired after T1.")))

    # Q5 – Impact assessment
    qa.append(dict(
        question="What are the potential implications of the observed changes?",
        answer=(f"The {info['name']} observed between T1 and T2 has significant implications: "
                f"{info['impact']}. The original {ctx} in T1 has been affected by "
                f"{info['desc']}, as evidenced by {info['t2_desc']} visible in T2.")))

    return qa


# Build full change-detection QA dataset
all_qa = []
for pair in tqdm(bitemporal_pairs, desc="Generating change QA"):
    for qa in generate_change_qa(pair):
        all_qa.append(dict(
            t1_path=pair["t1_path"],
            t2_path=pair["t2_path"],
            cls=pair["cls"],
            change_type=pair["change_type"],
            pair_id=pair["pair_id"],
            question=qa["question"],
            answer=qa["answer"],
        ))

random.shuffle(all_qa)
_split = int(len(all_qa) * (1 - CONFIG["val_split"]))
train_data = all_qa[:_split]
val_data   = all_qa[_split:]

print(f"\n📊 Change-Detection QA Dataset")
print(f"   Total    : {len(all_qa)}")
print(f"   Train    : {len(train_data)}")
print(f"   Val      : {len(val_data)}")
print(f"   Pairs    : {len(bitemporal_pairs)}")

for _name, _data in [("train_qa_pairs.json", train_data), ("val_qa_pairs.json", val_data)]:
    with open(f"{CONFIG['drive_output']}/dataset/{_name}", "w") as f:
        json.dump(_data, f, indent=2)
print("💾 Saved to Drive")

In [ ]:
# ── 2e  Dataset Distribution & Sample Pairs ─────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Change type distribution
_ct = defaultdict(int)
for s in all_qa:
    _ct[s["change_type"]] += 1
_types = sorted(_ct); _cnts = [_ct[t] for t in _types]
axes[0].barh(_types, _cnts, color=sns.color_palette("husl", len(_types)))
axes[0].set_xlabel("QA Pairs"); axes[0].set_title("Per Change-Type QA Count", fontweight="bold")

# Question type distribution
_qt = defaultdict(int)
for s in all_qa:
    if "What changes occurred" in s["question"]:
        _qt["Change\nDetection"] += 1
    elif "type of land-cover" in s["question"]:
        _qt["Change\nClassification"] += 1
    elif "extensive" in s["question"]:
        _qt["Change\nExtent"] += 1
    elif "recent" in s["question"] or "evidence" in s["question"]:
        _qt["Temporal\nReasoning"] += 1
    elif "implications" in s["question"]:
        _qt["Impact\nAssessment"] += 1
axes[1].bar(_qt.keys(), _qt.values(), color=sns.color_palette("Set2", len(_qt)))
axes[1].set_ylabel("Count"); axes[1].set_title("Question Type Distribution", fontweight="bold")
plt.setp(axes[1].xaxis.get_majorticklabels(), fontsize=8)

# Train / val
axes[2].bar(["Train", "Val"], [len(train_data), len(val_data)],
            color=["#2ecc71", "#f39c12"])
axes[2].set_ylabel("QA Pairs"); axes[2].set_title("Train / Val Split", fontweight="bold")
for i, v in enumerate([len(train_data), len(val_data)]):
    axes[2].text(i, v + 3, str(v), ha="center", fontweight="bold")

plt.suptitle("Change-Detection Dataset Overview", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{CONFIG['drive_output']}/results/dataset_distribution.png", bbox_inches="tight")
plt.show()

# Sample bi-temporal pairs — T1 (top) + T2 (bottom) across change types
_sample_types = list(CHANGE_FUNCTIONS.keys())
fig, axes = plt.subplots(2, len(_sample_types), figsize=(20, 7))
fig.suptitle("Sample Bi-Temporal Pairs — T1 Before (top) · T2 After (bottom)",
             fontsize=14, fontweight="bold")
for i, ct in enumerate(_sample_types):
    pair = next(p for p in bitemporal_pairs if p["change_type"] == ct)
    axes[0, i].imshow(Image.open(pair["t1_path"]))
    axes[0, i].set_title(f"{ct}\n(T1: {pair['cls']})", fontsize=8); axes[0, i].axis("off")
    axes[1, i].imshow(Image.open(pair["t2_path"]))
    axes[1, i].set_title(f"(T2: {ct})", fontsize=8); axes[1, i].axis("off")
axes[0, 0].set_ylabel("📅 T1 (Before)", fontsize=11, fontweight="bold")
axes[1, 0].set_ylabel("📅 T2 (After)", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['drive_output']}/results/sample_pairs.png", bbox_inches="tight")
plt.show()
print("✅ Distribution & sample pair plots saved")

---
## 3 · Data Preprocessing & DataLoaders

In [ ]:
from transformers import Blip2Processor

print("=" * 60)
print("⚙️  STEP 4 : Data Pipeline (Bi-Temporal)")
print("=" * 60)

processor = Blip2Processor.from_pretrained(CONFIG["model_name"])


class BiTemporalDataset(Dataset):
    """Bi-temporal dataset — returns BOTH T1 + T2 images per sample."""

    def __init__(self, data, processor, max_length=256):
        self.data = data
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Load both images
        t1_img = Image.open(item["t1_path"]).convert("RGB")
        t2_img = Image.open(item["t2_path"]).convert("RGB")

        q, a = item["question"], item["answer"]

        # Process both images through BLIP-2 image processor
        t1_enc = self.processor(images=t1_img, text="", return_tensors="pt")
        t2_enc = self.processor(images=t2_img, text="", return_tensors="pt")

        # Process text
        prompt = f"Question: {q} Answer: {a}"
        txt_enc = self.processor(
            images=t1_img, text=prompt, return_tensors="pt",
            padding="max_length", max_length=self.max_length, truncation=True,
        )

        t1_pv = t1_enc["pixel_values"].squeeze(0)
        t2_pv = t2_enc["pixel_values"].squeeze(0)
        ids   = txt_enc["input_ids"].squeeze(0)
        am    = txt_enc["attention_mask"].squeeze(0)

        # Labels: mask prompt tokens, keep only answer for loss
        labels = ids.clone()
        _prompt_only = f"Question: {q} Answer:"
        _plen = len(self.processor.tokenizer(_prompt_only, add_special_tokens=True)["input_ids"])
        labels[:_plen] = -100
        labels[am == 0] = -100

        return dict(
            t1_pixel_values=t1_pv,
            t2_pixel_values=t2_pv,
            input_ids=ids,
            attention_mask=am,
            labels=labels,
            question=q, answer=a,
            cls=item["cls"],
            change_type=item["change_type"],
        )


def bitemporal_collate_fn(batch):
    return dict(
        t1_pixel_values =torch.stack([b["t1_pixel_values"] for b in batch]),
        t2_pixel_values =torch.stack([b["t2_pixel_values"] for b in batch]),
        input_ids       =torch.stack([b["input_ids"]       for b in batch]),
        attention_mask  =torch.stack([b["attention_mask"]   for b in batch]),
        labels          =torch.stack([b["labels"]           for b in batch]),
        questions       =[b["question"]    for b in batch],
        answers         =[b["answer"]      for b in batch],
        classes         =[b["cls"]         for b in batch],
        change_types    =[b["change_type"] for b in batch],
    )


train_ds = BiTemporalDataset(train_data, processor, CONFIG["max_length"])
val_ds   = BiTemporalDataset(val_data,   processor, CONFIG["max_length"])

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"],
                          shuffle=True,  collate_fn=bitemporal_collate_fn,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"],
                          shuffle=False, collate_fn=bitemporal_collate_fn,
                          num_workers=0, pin_memory=True)

_sb = next(iter(train_loader))
print(f"\n📊 Bi-Temporal Loaders ready")
print(f"   Train batches     : {len(train_loader)}")
print(f"   Val   batches     : {len(val_loader)}")
print(f"   t1_pixel_values   : {_sb['t1_pixel_values'].shape}")
print(f"   t2_pixel_values   : {_sb['t2_pixel_values'].shape}")
print(f"   input_ids         : {_sb['input_ids'].shape}")
print("✅ Bi-temporal data pipeline ready")

---
## 4 · Model Setup — Dual-Stream BLIP-2 + QLoRA

```
  ┌──────────┐        ┌──────────┐
  │  T1 img  │        │  T2 img  │
  │ (before) │        │ (after)  │
  └────┬─────┘        └────┬─────┘
       │                    │
  ┌────▼─────┐        ┌────▼─────┐
  │  Shared  │(frozen)│  Shared  │  (same weights)
  │   ViT    │        │   ViT    │
  └────┬─────┘        └────┬─────┘
       │                    │
  ┌────▼─────┐        ┌────▼─────┐
  │  Shared  │(frozen)│  Shared  │  (same weights)
  │ Q-Former │        │ Q-Former │
  └────┬─────┘        └────┬─────┘
       │                    │
  [32 tokens]          [32 tokens]
       │                    │
       └────────┬───────────┘
                │
           ┌────▼────┐
           │ CONCAT  │  →  [64 visual tokens]
           └────┬────┘
                │
           ┌────▼────┐
           │ Language │
           │Projection│
           └────┬────┘
                │
           ┌────▼────┐
           │  OPT    │ ← LoRA adapters (trainable)
           │  2.7B   │
           └────┬────┘
                │
           [Change Detection Answer]
```

In [ ]:
from transformers import Blip2ForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

print("=" * 60)
print("🧠  STEP 5 : Loading Dual-Stream BLIP-2 with QLoRA")
print("=" * 60)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("  Loading BLIP-2 base model (2-3 min) …")
base_model = Blip2ForConditionalGeneration.from_pretrained(
    CONFIG["model_name"],
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
base_model = prepare_model_for_kbit_training(base_model)

# Freeze vision encoder + Q-Former
for p in base_model.vision_model.parameters():
    p.requires_grad = False
for p in base_model.qformer.parameters():
    p.requires_grad = False
print("  ✅ Vision encoder & Q-Former frozen")

# Apply LoRA to language model
lora_cfg = LoraConfig(
    r=CONFIG["lora_rank"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
base_model = get_peft_model(base_model, lora_cfg)

_train_p, _total_p = base_model.get_nb_trainable_parameters()
print(f"\n📊 Parameters (before dual-stream wrapper)")
print(f"   Total      : {_total_p:>12,}")
print(f"   Trainable  : {_train_p:>12,}  ({100*_train_p/_total_p:.2f}%)")

In [ ]:
# ── Dual-Stream Wrapper ─────────────────────────────────────────────────────

class DualStreamBLIP2(nn.Module):
    """
    Dual-stream wrapper for BLIP-2 that processes T1 (before) + T2 (after)
    images independently through the shared frozen ViT + Q-Former, concatenates
    their 32-token query outputs → 64 visual tokens, projects to LM space,
    and feeds to the LoRA-adapted OPT language model for change detection VQA.
    """

    def __init__(self, blip2_model):
        super().__init__()
        self.blip2 = blip2_model

    def _encode_image(self, pixel_values):
        """Run a single image through ViT + Q-Former → query outputs."""
        bm = self.blip2
        if hasattr(bm, "base_model"):  # unwrap PEFT
            bm = bm.base_model.model if hasattr(bm, "base_model") else bm
        # Access underlying model components
        _m = bm
        while hasattr(_m, "model"):
            _m = _m.model

        vision_outputs = _m.vision_model(pixel_values=pixel_values, return_dict=True)
        image_embeds   = vision_outputs.last_hidden_state

        # Q-Former forward
        image_attention_mask = torch.ones(image_embeds.size()[:-1],
                                          dtype=torch.long, device=image_embeds.device)
        query_tokens = _m.query_tokens.expand(image_embeds.shape[0], -1, -1)
        query_outputs = _m.qformer(
            query_embeds=query_tokens,
            encoder_hidden_states=image_embeds,
            encoder_attention_mask=image_attention_mask,
            return_dict=True,
        )
        return query_outputs.last_hidden_state  # (batch, 32, qformer_hidden)

    def _get_lm_components(self):
        """Get language projection and language model from the BLIP-2 model."""
        _m = self.blip2
        while hasattr(_m, "model"):
            _m = _m.model
        return _m.language_projection, _m.language_model

    def forward(self, t1_pixel_values, t2_pixel_values,
                input_ids, attention_mask, labels=None):
        """
        Forward pass with dual-stream temporal fusion.

        1. Encode T1 (before) → 32 query tokens
        2. Encode T2 (after)  → 32 query tokens
        3. Concatenate        → 64 query tokens
        4. Project to LM embedding space
        5. Prepend to text embeddings → OPT language model
        """
        # Stream 1: T1 (before)
        t1_query = self._encode_image(t1_pixel_values)  # (B, 32, H)
        # Stream 2: T2 (after)
        t2_query = self._encode_image(t2_pixel_values)  # (B, 32, H)

        # ── TEMPORAL FUSION: Concatenate along sequence dimension ──
        fused_query = torch.cat([t1_query, t2_query], dim=1)  # (B, 64, H)

        # Project to language model hidden size
        language_projection, language_model = self._get_lm_components()
        language_model_inputs = language_projection(fused_query)  # (B, 64, LM_H)

        # Build attention mask for visual tokens (all ones)
        vis_attn = torch.ones(language_model_inputs.size()[:-1],
                              dtype=torch.long, device=language_model_inputs.device)

        # Get text embeddings from LM
        if hasattr(language_model, "model"):  # OPT structure
            text_embeds = language_model.model.decoder.embed_tokens(input_ids)
        else:
            text_embeds = language_model.get_input_embeddings()(input_ids)

        # Concatenate: [visual tokens (64)] + [text tokens]
        inputs_embeds  = torch.cat([language_model_inputs, text_embeds.to(language_model_inputs.dtype)], dim=1)
        full_attn_mask = torch.cat([vis_attn, attention_mask], dim=1)

        if labels is not None:
            # Pad labels with -100 for the 64 visual token positions
            vis_labels = torch.full((labels.shape[0], fused_query.shape[1]),
                                     -100, dtype=labels.dtype, device=labels.device)
            full_labels = torch.cat([vis_labels, labels], dim=1)
        else:
            full_labels = None

        outputs = language_model(
            inputs_embeds=inputs_embeds,
            attention_mask=full_attn_mask,
            labels=full_labels,
            return_dict=True,
        )
        return outputs

    @torch.no_grad()
    def generate(self, t1_pixel_values, t2_pixel_values,
                 input_ids, attention_mask, **gen_kwargs):
        """Generate text for inference with mixed-precision autocast support."""
        _device_type = "cuda" if torch.cuda.is_available() else "cpu"
        with torch.autocast(device_type=_device_type):
            t1_query = self._encode_image(t1_pixel_values)
            t2_query = self._encode_image(t2_pixel_values)
            fused_query = torch.cat([t1_query, t2_query], dim=1)

            language_projection, language_model = self._get_lm_components()
            language_model_inputs = language_projection(fused_query)

            vis_attn = torch.ones(language_model_inputs.size()[:-1],
                                  dtype=torch.long, device=language_model_inputs.device)

            if hasattr(language_model, "model"):
                text_embeds = language_model.model.decoder.embed_tokens(input_ids)
            else:
                text_embeds = language_model.get_input_embeddings()(input_ids)

            inputs_embeds  = torch.cat([language_model_inputs, text_embeds.to(language_model_inputs.dtype)], dim=1)
            full_attn_mask = torch.cat([vis_attn, attention_mask], dim=1)

            outputs = language_model.generate(
                inputs_embeds=inputs_embeds,
                attention_mask=full_attn_mask,
                **gen_kwargs,
            )
            return outputs


# Instantiate the dual-stream model
model = DualStreamBLIP2(base_model)
model = model.to(device) if not hasattr(base_model, "hf_device_map") else model

print("\n🔀 Dual-Stream BLIP-2 Architecture (Temporal):")
print("   T1 (before) → ViT → Q-Former → [32 tokens]")
print("   T2 (after)  → ViT → Q-Former → [32 tokens]")
print("   Concatenated → [64 tokens] → Language Projection → OPT-2.7B (LoRA)")

if torch.cuda.is_available():
    print(f"\n💾 GPU Memory : {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")
print("\n✅ Dual-stream model ready")

---
## 5 · Training Loop

In [ ]:
from transformers import get_cosine_schedule_with_warmup

print("=" * 60)
print("🏋️  STEP 6 : Training (Change Detection)")
print("=" * 60)

# Collect only trainable params for optimizer
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=CONFIG["learning_rate"],
                              weight_decay=0.01, betas=(0.9, 0.999))

_total_steps  = len(train_loader) * CONFIG["num_epochs"] // CONFIG["grad_accum_steps"]
_warmup_steps = int(_total_steps * CONFIG["warmup_ratio"])
scheduler = get_cosine_schedule_with_warmup(optimizer, _warmup_steps, _total_steps)

scaler = torch.cuda.amp.GradScaler()
training_log = []
best_val_loss = float("inf")
patience_ctr  = 0
best_epoch    = 0
all_lrs       = []

print(f"   Epochs         : {CONFIG['num_epochs']}")
print(f"   Batch size     : {CONFIG['batch_size']}  (eff. {CONFIG['batch_size']*CONFIG['grad_accum_steps']})")
print(f"   Optim. steps   : {_total_steps}")
print(f"   Warmup steps   : {_warmup_steps}")
print(f"   Learning rate  : {CONFIG['learning_rate']}\n")

t0 = time.time()

for epoch in range(CONFIG["num_epochs"]):
    # ── Train ────────────────────────────────────────────────────
    model.train()
    _tloss, _tn = 0.0, 0
    optimizer.zero_grad()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['num_epochs']} [Train]")
    for step, batch in enumerate(pbar):
        with torch.cuda.amp.autocast():
            out = model(
                t1_pixel_values=batch["t1_pixel_values"].to(device),
                t2_pixel_values=batch["t2_pixel_values"].to(device),
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                labels=batch["labels"].to(device),
            )
            loss = out.loss / CONFIG["grad_accum_steps"]
        scaler.scale(loss).backward()

        if (step + 1) % CONFIG["grad_accum_steps"] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            all_lrs.append(scheduler.get_last_lr()[0])

        _tloss += loss.item() * CONFIG["grad_accum_steps"]
        _tn += 1
        pbar.set_postfix(loss=f"{_tloss/_tn:.4f}")

    avg_train = _tloss / _tn

    # ── Validate ─────────────────────────────────────────────────
    model.eval()
    _vloss, _vn = 0.0, 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{CONFIG['num_epochs']} [Val]"):
            with torch.cuda.amp.autocast():
                out = model(
                    t1_pixel_values=batch["t1_pixel_values"].to(device),
                    t2_pixel_values=batch["t2_pixel_values"].to(device),
                    input_ids=batch["input_ids"].to(device),
                    attention_mask=batch["attention_mask"].to(device),
                    labels=batch["labels"].to(device),
                )
            _vloss += out.loss.item()
            _vn += 1
    avg_val = _vloss / _vn

    training_log.append(dict(epoch=epoch+1, train_loss=avg_train, val_loss=avg_val,
                             lr=optimizer.param_groups[0]["lr"]))
    print(f"\n  📈 Epoch {epoch+1:>2d}  train_loss={avg_train:.4f}  val_loss={avg_val:.4f}"
          f"  lr={optimizer.param_groups[0]['lr']:.2e}")

    # ── Checkpointing ─────────────────────────────────────────────
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        best_epoch = epoch + 1
        patience_ctr = 0
        # Save LoRA adapter weights
        base_model.save_pretrained(f"{CONFIG['drive_output']}/checkpoints")
        processor.save_pretrained(f"{CONFIG['drive_output']}/checkpoints/processor")
        print(f"  💾 Best model saved (epoch {best_epoch})")
    else:
        patience_ctr += 1
        if patience_ctr >= CONFIG["early_stop_patience"]:
            print(f"\n  ⏹️  Early stopping (patience {CONFIG['early_stop_patience']})")
            break

elapsed = time.time() - t0
print(f"\n⏱️  Training completed in {elapsed/60:.1f} min")
print(f"   Best epoch: {best_epoch}  val_loss: {best_val_loss:.4f}")

# Save training log
with open(f"{CONFIG['drive_output']}/results/training_log.json", "w") as f:
    json.dump(training_log, f, indent=2)
print("✅ Training complete")

---
## 6 · Evaluation & Metrics

In [ ]:
print("=" * 60)
print("📊  STEP 7 : Evaluation")
print("=" * 60)

# Reuse trained model from memory (or load from Drive)
from peft import PeftModel
try:
    if 'model' in globals() and model is not None:
        eval_model = model
        print("  ✅ Using trained model from memory")
    else:
        _base = Blip2ForConditionalGeneration.from_pretrained(
            CONFIG["model_name"], quantization_config=bnb_config,
            device_map="auto", torch_dtype=torch.float16,
        )
        _base = PeftModel.from_pretrained(_base, f"{CONFIG['drive_output']}/checkpoints")
        eval_model = DualStreamBLIP2(_base)
        print("  ✅ Best checkpoint loaded from Drive")
except Exception as e:
    print(f"  ⚠️ Checkpoint load exception ({e}); using current weights")
    eval_model = model

eval_model.eval()

# Generate predictions on validation set
predictions = []
with torch.no_grad(), torch.cuda.amp.autocast():
    for batch in tqdm(val_loader, desc="Generating predictions"):
        # Build inference prompt (question only, no answer)
        for i in range(len(batch["questions"])):
            q = batch["questions"][i]
            prompt = f"Question: {q} Answer:"
            enc = processor.tokenizer(prompt, return_tensors="pt",
                                       padding="max_length", max_length=CONFIG["max_length"],
                                       truncation=True)

            gen = eval_model.generate(
                t1_pixel_values=batch["t1_pixel_values"][i:i+1].to(device),
                t2_pixel_values=batch["t2_pixel_values"][i:i+1].to(device),
                input_ids=enc["input_ids"].to(device),
                attention_mask=enc["attention_mask"].to(device),
                max_new_tokens=128, do_sample=False, num_beams=3,
                repetition_penalty=1.2,
            )
            pred = processor.batch_decode(gen, skip_special_tokens=True)[0].strip()
            if "Answer:" in pred:
                pred = pred.split("Answer:")[-1].strip()

            predictions.append(dict(
                question=q, ground_truth=batch["answers"][i],
                prediction=pred, cls=batch["classes"][i],
                change_type=batch["change_types"][i]))

print(f"  Generated {len(predictions)} predictions")

# Compute metrics
import evaluate as hf_evaluate

bleu_metric  = hf_evaluate.load("bleu")
rouge_metric = hf_evaluate.load("rouge")

refs  = [p["ground_truth"] for p in predictions]
preds = [p["prediction"] for p in predictions]

bleu1 = bleu_metric.compute(predictions=preds, references=[[r] for r in refs],
                             max_order=1)["bleu"]
bleu4 = bleu_metric.compute(predictions=preds, references=[[r] for r in refs],
                             max_order=4)["bleu"]
rouge = rouge_metric.compute(predictions=preds, references=refs)
rougeL = rouge["rougeL"]

# Per-prediction ROUGE-L for distribution analysis
_rouge_per = []
for p, r in zip(preds, refs):
    _s = rouge_metric.compute(predictions=[p], references=[r])["rougeL"]
    _rouge_per.append(_s)

# Per-change-type metrics
per_ct_metrics = {}
for ct in CHANGE_FUNCTIONS.keys():
    ct_preds = [p for p in predictions if p["change_type"] == ct]
    if ct_preds:
        ct_refs  = [p["ground_truth"] for p in ct_preds]
        ct_pred  = [p["prediction"] for p in ct_preds]
        try:
            ct_b1 = bleu_metric.compute(predictions=ct_pred, references=[[r] for r in ct_refs], max_order=1)["bleu"]
        except Exception:
            ct_b1 = 0.0
        try:
            ct_rl = rouge_metric.compute(predictions=ct_pred, references=ct_refs)["rougeL"]
        except Exception:
            ct_rl = 0.0
        per_ct_metrics[ct] = {"BLEU-1": round(ct_b1, 4), "ROUGE-L": round(ct_rl, 4), "count": len(ct_preds)}

metrics = {"BLEU-1": round(bleu1, 4), "BLEU-4": round(bleu4, 4),
           "ROUGE-L": round(rougeL, 4),
           "best_epoch": best_epoch, "best_val_loss": round(best_val_loss, 4),
           "training_time_min": round(elapsed / 60, 1),
           "num_pairs": len(bitemporal_pairs), "num_qa": len(all_qa),
           "per_change_type": per_ct_metrics}

print(f"\n📊 Change-Detection VQA Metrics")
print(f"   BLEU-1  = {metrics['BLEU-1']}")
print(f"   BLEU-4  = {metrics['BLEU-4']}")
print(f"   ROUGE-L = {metrics['ROUGE-L']}")
print(f"\n   Per Change Type:")
for ct, m in per_ct_metrics.items():
    print(f"     {ct:20s}  BLEU-1={m['BLEU-1']:.4f}  ROUGE-L={m['ROUGE-L']:.4f}  (n={m['count']})")

with open(f"{CONFIG['drive_output']}/results/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# Save predictions CSV
with open(f"{CONFIG['drive_output']}/results/predictions.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["question", "ground_truth", "prediction", "cls", "change_type", "rouge_l"])
    w.writeheader()
    for p, rl in zip(predictions, _rouge_per):
        w.writerow({**p, "rouge_l": round(rl, 4)})

print("💾 Metrics & predictions saved")

---
## 7 · Visualizations

In [ ]:
# ── 7a  Loss Curves & LR Schedule ───────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_x = [l["epoch"] for l in training_log]
ax1.plot(epochs_x, [l["train_loss"] for l in training_log], "o-", label="Train", c="#3498db", lw=2)
ax1.plot(epochs_x, [l["val_loss"] for l in training_log],   "s-", label="Val",   c="#e74c3c", lw=2)
ax1.axvline(best_epoch, ls="--", c="green", alpha=0.6, label=f"Best (ep {best_epoch})")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Training & Validation Loss", fontweight="bold"); ax1.legend(); ax1.grid(True, alpha=0.3)

if all_lrs:
    ax2.plot(all_lrs, c="#9b59b6", lw=1.5)
    ax2.set_xlabel("Optimizer Step"); ax2.set_ylabel("Learning Rate")
    ax2.set_title("Cosine LR Schedule", fontweight="bold"); ax2.grid(True, alpha=0.3)
    ax2.ticklabel_format(style="sci", axis="y", scilimits=(0, 0))

plt.suptitle("Change-Detection Training Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['drive_output']}/results/loss_lr_curves.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── 7b  Overall Metric Bar Charts ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
_mnames = ["BLEU-1", "BLEU-4", "ROUGE-L"]
_mvals  = [metrics[m] for m in _mnames]
bars = ax.bar(_mnames, _mvals, color=["#3498db", "#2ecc71", "#e74c3c"], width=0.5, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, _mvals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{val:.4f}", ha="center", fontweight="bold", fontsize=12)
ax.set_ylim(0, max(_mvals) * 1.3 + 0.05)
ax.set_ylabel("Score"); ax.set_title("Change-Detection VQA — Overall Metrics", fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{CONFIG['drive_output']}/results/metrics_summary.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── 7c  Per-Change-Type Metrics ─────────────────────────────────────────────
if per_ct_metrics:
    _ct_names = sorted(per_ct_metrics.keys())
    _ct_b1    = [per_ct_metrics[ct]["BLEU-1"] for ct in _ct_names]
    _ct_rl    = [per_ct_metrics[ct]["ROUGE-L"] for ct in _ct_names]

    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(_ct_names)); w = 0.35
    bars1 = ax.bar(x - w/2, _ct_b1, w, label="BLEU-1", color="#3498db")
    bars2 = ax.bar(x + w/2, _ct_rl, w, label="ROUGE-L", color="#e74c3c")
    ax.set_xticks(x); ax.set_xticklabels([ct.replace("_", "\n") for ct in _ct_names], fontsize=9)
    ax.set_ylabel("Score"); ax.set_title("Metrics by Change Type", fontweight="bold")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    for bars in [bars1, bars2]:
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax.text(bar.get_x() + bar.get_width()/2, h + 0.005, f"{h:.3f}",
                        ha="center", fontsize=7, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{CONFIG['drive_output']}/results/change_type_metrics.png", bbox_inches="tight")
    plt.show()
    print("✅ Per-change-type metrics saved")

In [ ]:
# ── 7d  Paired Prediction Grid ──────────────────────────────────────────────
# Show T1+T2 pair → question → ground truth vs predicted answer
n_show = min(3, len(predictions))
fig, axes = plt.subplots(n_show, 2, figsize=(12, 5 * n_show))
if n_show == 1:
    axes = axes.reshape(1, -1)

# Select predictions from different change types for diversity
_seen_ct = set()
_diverse_preds = []
for p in predictions:
    if p["change_type"] not in _seen_ct and len(_diverse_preds) < n_show:
        _diverse_preds.append(p)
        _seen_ct.add(p["change_type"])
# Fill remaining with any predictions
while len(_diverse_preds) < n_show:
    _diverse_preds.append(predictions[len(_diverse_preds)])

for i, p in enumerate(_diverse_preds):
    # Find the bi-temporal pair
    pair = None
    for s in bitemporal_pairs:
        if s["change_type"] == p["change_type"] and s["cls"] == p["cls"]:
            pair = s; break
    if pair is None:
        pair = bitemporal_pairs[0]

    axes[i, 0].imshow(Image.open(pair["t1_path"]))
    axes[i, 0].set_title(f"📅 T1 (Before) — {p['cls']}\n[{p['change_type']}]", fontsize=9, fontweight="bold")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(Image.open(pair["t2_path"]))
    axes[i, 1].set_title(f"📅 T2 (After) — {p['cls']}\n[{p['change_type']}]", fontsize=9, fontweight="bold")
    axes[i, 1].axis("off")

    _q  = p["question"][:80]
    _gt = p["ground_truth"][:80]
    _pr = p["prediction"][:80]
    fig.text(0.5, 1.0 - (i / n_show) - 0.01,
             f"Q: {_q}…\nGT: {_gt}…\nPred: {_pr}…",
             ha="center", fontsize=8, style="italic",
             bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

plt.suptitle("Change-Detection VQA — Sample Predictions", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{CONFIG['drive_output']}/results/predictions_grid.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── 7e  Score Distribution & Best / Worst ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(_rouge_per, bins=15, color="#3498db", edgecolor="white", alpha=0.8)
axes[0].axvline(np.mean(_rouge_per), color="red", ls="--", lw=2, label=f"Mean={np.mean(_rouge_per):.3f}")
axes[0].set_xlabel("ROUGE-L"); axes[0].set_ylabel("Count")
axes[0].set_title("Per-Sample ROUGE-L Distribution", fontweight="bold")
axes[0].legend()

# Best and worst examples
_sorted = sorted(zip(_rouge_per, predictions), key=lambda x: x[0])
_worst = _sorted[:2]
_best  = _sorted[-2:]
_examples = _worst + _best
_labels   = ["Worst 1", "Worst 2", "Best 2", "Best 1"]
_colors   = ["#e74c3c", "#e74c3c", "#2ecc71", "#2ecc71"]
axes[1].barh(_labels, [e[0] for e in _examples], color=_colors)
axes[1].set_xlabel("ROUGE-L"); axes[1].set_title("Best / Worst Predictions", fontweight="bold")
for i, (rl, p) in enumerate(_examples):
    axes[1].text(rl + 0.01, i, f"{p['change_type']}: {p['question'][:30]}…", fontsize=7, va="center")

plt.suptitle("Change-Detection VQA — Score Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['drive_output']}/results/score_distribution.png", bbox_inches="tight")
plt.show()
print("✅ All visualizations saved")

---
## 8 · Model Export & Training Report

In [ ]:
report = f"""# SatQuery AI — Model 3 : Multi-Temporal Change Detection VQA Training Report

**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
**Team:** Spectra | SIH 2026

---

## Architecture
- **Model:** Dual-Stream BLIP-2 OPT-2.7B (4-bit quantized)
- **Innovation:** Shared ViT + Q-Former processes T1 (before) & T2 (after)
  independently, query outputs concatenated (32+32=64 visual tokens) → LoRA OPT
- **Fine-tuning:** QLoRA (rank {CONFIG['lora_rank']}, ~{_train_p:,} trainable params)
- **Training Time:** {elapsed/60:.1f} minutes on Colab T4

## Dataset
- **Bi-Temporal Pairs:** {len(bitemporal_pairs)}
- **T1 Source:** EuroSAT (Sentinel-2), {len(EUROSAT_CLASSES)} classes
- **T2 Source:** Synthesized with 6 change types (urbanisation, deforestation, flooding, seasonal crop, fire/burn, no-change)
- **Change-Detection QA Pairs:** {len(all_qa)} ({len(train_data)} train / {len(val_data)} val)
- **Question Types:** 5 temporal reasoning categories

## Results
| Metric | Score |
|--------|-------|
| BLEU-1 | {metrics['BLEU-1']} |
| BLEU-4 | {metrics['BLEU-4']} |
| ROUGE-L | {metrics['ROUGE-L']} |

- Best epoch: {best_epoch} (val_loss = {best_val_loss:.4f})

### Per Change Type
| Change Type | BLEU-1 | ROUGE-L | Samples |
|-------------|--------|---------|---------|
"""

for ct, m in sorted(per_ct_metrics.items()):
    report += f"| {ct} | {m['BLEU-1']} | {m['ROUGE-L']} | {m['count']} |\n"

report += f"""
## Key Innovation
- **Temporal change detection via visual language model:** The dual-stream
  architecture captures temporal differences by encoding T1 and T2 through
  the same shared vision backbone, allowing the language model to reason
  about **what changed, when, how much, and what the implications are**.
- Model generates **natural-language** answers requiring **temporal reasoning**
  across 6 distinct change categories.
"""

with open(f"{CONFIG['drive_output']}/report/training_report.md", "w") as f:
    f.write(report)

print("✅ Training report saved")
print("\n" + "=" * 60)
print("🎉  PIPELINE COMPLETE")
print("=" * 60)
print(f"\n📁 All outputs → {CONFIG['drive_output']}")
print(f"\n   BLEU-1  = {metrics['BLEU-1']}")
print(f"   BLEU-4  = {metrics['BLEU-4']}")
print(f"   ROUGE-L = {metrics['ROUGE-L']}")
print(f"   Best epoch {best_epoch}   val_loss={best_val_loss:.4f}")

---
## 9 · Interactive Change Detection Demo

Test the fine-tuned model with any bi-temporal pair + question.

In [ ]:
def satquery_change_detect(t1_path, t2_path, question):
    """Run change-detection VQA inference on a T1 + T2 bi-temporal pair."""
    t1_img = Image.open(t1_path).convert("RGB")
    t2_img = Image.open(t2_path).convert("RGB")

    # Process images
    t1_enc = processor(images=t1_img, text="", return_tensors="pt")
    t2_enc = processor(images=t2_img, text="", return_tensors="pt")

    # Process prompt
    prompt = f"Question: {question} Answer:"
    txt_enc = processor.tokenizer(prompt, return_tensors="pt",
                                   padding="max_length", max_length=CONFIG["max_length"],
                                   truncation=True)

    with torch.no_grad(), torch.cuda.amp.autocast():
        gen = eval_model.generate(
            t1_pixel_values=t1_enc["pixel_values"].to(device),
            t2_pixel_values=t2_enc["pixel_values"].to(device),
            input_ids=txt_enc["input_ids"].to(device),
            attention_mask=txt_enc["attention_mask"].to(device),
            max_new_tokens=128, do_sample=False, num_beams=3,
            repetition_penalty=1.2,
        )
    ans = processor.batch_decode(gen, skip_special_tokens=True)[0].strip()
    if "Answer:" in ans:
        ans = ans.split("Answer:")[-1].strip()
    return ans


# ── Demo queries ─────────────────────────────────────────────────────────────
print("📅→📅  SatQuery AI — Change Detection VQA Demo")
print("=" * 55)

# Select one pair per change type for demo
_demo_pairs = []
_seen = set()
for p in bitemporal_pairs:
    if p["change_type"] not in _seen:
        _demo_pairs.append(p)
        _seen.add(p["change_type"])
_demo_pairs = _demo_pairs[:4]  # limit to 4 for display

_demo_questions = [
    "What changes occurred in this area between the two observation times?",
    "What type of land-cover change has occurred between T1 and T2?",
    "How extensive is the change between T1 and T2?",
    "What are the potential implications of the observed changes?",
]

fig, axes = plt.subplots(len(_demo_pairs), 2, figsize=(10, 5 * len(_demo_pairs)))
if len(_demo_pairs) == 1:
    axes = axes.reshape(1, -1)

for i, pair in enumerate(_demo_pairs):
    q = _demo_questions[i % len(_demo_questions)]
    ans = satquery_change_detect(pair["t1_path"], pair["t2_path"], q)

    axes[i, 0].imshow(Image.open(pair["t1_path"]))
    axes[i, 0].set_title(f"📅 T1 (Before) — {pair['cls']}", fontsize=10); axes[i, 0].axis("off")
    axes[i, 1].imshow(Image.open(pair["t2_path"]))
    axes[i, 1].set_title(f"📅 T2 (After) — {pair['change_type']}", fontsize=10); axes[i, 1].axis("off")

    print(f"\n🔄 [{pair['change_type']}] {pair['cls']}")
    print(f"   Q: {q}")
    print(f"   A: {ans}")

plt.suptitle("SatQuery AI — Change Detection VQA Demo", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['drive_output']}/results/demo_outputs.png", bbox_inches="tight")
plt.show()

print("\n✅ Demo complete — change detection model ready for SatQuery AI integration!")